# Finding 1 — UPI's Rise to Dominance
**Study:** Mapping UPI: Scale, Geography, and Competition in India's Payment Ecosystem
**Data:** NPCI Product Statistics (2016–2026), RBI Payment System Indicators (2019–2026)
**Analysis:** UPI value CAGR vs legacy rails, average ticket size evolution, network effect, UPI share of total digital payment value

In [ ]:
import subprocess, sys
for pkg in ["matplotlib", "seaborn", "pandas", "numpy", "sqlalchemy", "pymysql", "scipy"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats
from sqlalchemy import create_engine
from pathlib import Path

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.6)
plt.rcParams.update({
    "figure.dpi": 200, "savefig.dpi": 200,
    "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})
print("Ready.")

## Data Load — MySQL Source

In [ ]:
# MySQL connection — set your own credentials via environment or edit below.
DB_USER = "your_username"    # your MySQL username
DB_PASS = "your_password"    # your MySQL password
DB_HOST = "localhost"        # or your MySQL host
DB_NAME = "upi_data"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}?charset=utf8mb4"
)

# UPI aggregate — NPCI Product Statistics 2016–2026
upi_raw = pd.read_sql("""
    SELECT month_year, volume_mn, value_cr, banks_on_upi AS banks_live_on_upi
    FROM upi_aggregate
    ORDER BY month_year
""", engine)

# NEFT / RTGS — RBI Payment System Indicators 2019–2026
rbi_raw = pd.read_sql("""
    SELECT month_year, instrument, volume_mn, value_cr
    FROM rbi_instruments
    ORDER BY month_year
""", engine)

print(f"UPI rows  : {len(upi_raw)}  ({upi_raw['month_year'].min()} → {upi_raw['month_year'].max()})")
print(f"RBI rows  : {len(rbi_raw)}  instruments: {rbi_raw['instrument'].unique().tolist()}")
print(f"\nFirst 3 UPI rows:")
print(upi_raw.head(3).to_string(index=False))
print(f"\nFirst 4 RBI rows:")
print(rbi_raw.head(4).to_string(index=False))

## Ticket Size, Value Share, and CAGR

In [ ]:
# UPI ticket size: (value_cr / volume_mn) * 10 → rupees 
upi = upi_raw.copy()
upi["avg_ticket_rs"] = (upi["value_cr"] / upi["volume_mn"]) * 10
upi["date"] = pd.to_datetime(upi["month_year"] + "-01")
upi = upi[upi["volume_mn"] > 0].sort_values("date").reset_index(drop=True)

# Pivot RBI instruments → one row per month
rbi_pivot = rbi_raw.pivot_table(
    index="month_year", columns="instrument",
    values=["volume_mn", "value_cr"], aggfunc="sum"
).reset_index()
rbi_pivot.columns = ["_".join(c).strip("_") for c in rbi_pivot.columns]
rbi_pivot = rbi_pivot.rename(columns={
    "value_cr_NEFT": "neft_value",
    "value_cr_RTGS": "rtgs_value",
    "volume_mn_NEFT": "neft_vol",
    "volume_mn_RTGS": "rtgs_vol",
})

# Merge: only months where all three instruments have data
df = pd.merge(upi, rbi_pivot, on="month_year", how="inner")
df = df.dropna(subset=["value_cr", "neft_value", "rtgs_value"]).reset_index(drop=True)

df["upi_value"]           = df["value_cr"]
df["total_digital_value"] = df["upi_value"] + df["neft_value"] + df["rtgs_value"]
df["upi_share_pct"]       = df["upi_value"]  / df["total_digital_value"] * 100
df["neft_share_pct"]      = df["neft_value"] / df["total_digital_value"] * 100
df["rtgs_share_pct"]      = df["rtgs_value"] / df["total_digital_value"] * 100

print(f"Rows after merge (all 3 instruments): {len(df)}")
print(f"Range: {df['month_year'].min()} → {df['month_year'].max()}")
print()

# Key metrics 
first = df.iloc[0]
last  = df.iloc[-1]
n_years = (last["date"] - first["date"]).days / 365.25

ticket_first = first["avg_ticket_rs"]
ticket_last  = last["avg_ticket_rs"]
ticket_chg   = (ticket_last / ticket_first - 1) * 100

share_first  = first["upi_share_pct"]
share_last   = last["upi_share_pct"]

def cagr(first_val, last_val, years):
    return ((last_val / first_val) ** (1 / years) - 1) * 100

cagr_upi  = cagr(first["upi_value"],  last["upi_value"],  n_years)
cagr_neft = cagr(first["neft_value"], last["neft_value"], n_years)
cagr_rtgs = cagr(first["rtgs_value"], last["rtgs_value"], n_years)

# When did UPI first exceed NEFT in value?
upi_gt_neft = df[df["upi_value"] > df["neft_value"]]
crossover_neft = upi_gt_neft.iloc[0]["month_year"] if not upi_gt_neft.empty else "Not yet"

# When did UPI first exceed 10% share of digital payment value?
upi_10pct = df[df["upi_share_pct"] >= 10]
crossover_10pct = upi_10pct.iloc[0]["month_year"] if not upi_10pct.empty else "Not yet"

print(f"Ticket size first ({first['month_year']}): ₹{ticket_first:,.0f}")
print(f"Ticket size last  ({last['month_year']}): ₹{ticket_last:,.0f}  ({ticket_chg:+.0f}%)")
print(f"UPI share first  : {share_first:.1f}%  →  last: {share_last:.1f}%")
print(f"CAGR UPI value   : {cagr_upi:+.1f}% / yr")
print(f"CAGR NEFT value  : {cagr_neft:+.1f}% / yr")
print(f"CAGR RTGS value  : {cagr_rtgs:+.1f}% / yr")
print(f"UPI > NEFT value : {crossover_neft}")
print(f"UPI > 10% share  : {crossover_10pct}")

## Chart 1 — Ticket Size Evolution

In [ ]:
tk = upi[["date", "avg_ticket_rs"]].dropna().sort_values("date").reset_index(drop=True)
y_max = tk["avg_ticket_rs"].max() * 1.22   # headroom for annotations

fig, ax = plt.subplots(figsize=(20, 11))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# Gradient line 
x_num    = mdates.date2num(tk["date"].values)
y_val    = tk["avg_ticket_rs"].values
points   = np.array([x_num, y_val]).T.reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)
norm     = plt.Normalize(0, len(segments))
cmap     = LinearSegmentedColormap.from_list("blue_grad", ["#bfdbfe", "#1d4ed8"])
lc = LineCollection(segments, cmap=cmap, norm=norm, linewidth=3.5, zorder=4)
lc.set_array(np.arange(len(segments)))
ax.add_collection(lc)

# Set limits first so annotation y-positions are stable 
ax.set_xlim(tk["date"].min(), tk["date"].max())
ax.set_ylim(0, y_max)

# Reference lines
ax.axhline(500,  color="#94a3b8", lw=1.5, ls="--", alpha=0.7, label="₹500 reference")
ax.axhline(1000, color="#64748b", lw=1.5, ls="--", alpha=0.7, label="₹1,000 reference")

# Peak annotation (demonetisation spike)
peak_idx = int(tk["avg_ticket_rs"].idxmax())
peak_row = tk.iloc[peak_idx]
ax.annotate(
    f"Demonetisation spike: ₹{peak_row['avg_ticket_rs']:,.0f}\nearly adopters, high-value transfers",
    xy=(peak_row["date"], peak_row["avg_ticket_rs"]),
    xytext=(60, -14), textcoords="offset points",
    fontsize=14, color="#374151", ha="left", va="top",
    arrowprops=dict(arrowstyle="->", color="#ca8a04", lw=1.5,
                    connectionstyle="arc3,rad=0.25"),
    bbox=dict(boxstyle="round,pad=0.5", fc="#fef9c3",
              ec="#ca8a04", alpha=0.96, lw=1.2)
)

# Stabilisation annotation (post-2021 flat zone)
stab_candidates = tk[tk["date"] >= pd.to_datetime("2021-07-01")]
if not stab_candidates.empty:
    stab_row = stab_candidates.iloc[0]
    ax.annotate(
        "Mass adoption: ticket size stabilises\nas small payments dominate",
        xy=(stab_row["date"], stab_row["avg_ticket_rs"]),
        xytext=(14, 65), textcoords="offset points",
        fontsize=14, color="#374151", ha="left", va="bottom",
        arrowprops=dict(arrowstyle="->", color="#16a34a", lw=1.5,
                        connectionstyle="arc3,rad=-0.2"),
        bbox=dict(boxstyle="round,pad=0.5", fc="#f0fdf4",
                  ec="#16a34a", alpha=0.96, lw=1.2)
    )

# Axes
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"₹{y:,.0f}"))
ax.tick_params(axis="x", rotation=30, labelsize=14)
ax.tick_params(axis="y", labelsize=14)
ax.set_xlabel("Month", fontsize=16)
ax.set_ylabel("Avg Transaction Value (₹)", fontsize=16)
ax.set_title(
    "UPI Average Transaction Value —\n"
    "Demonetisation Spike to Mass Adoption Decline (2016–2026)",
    fontsize=20, fontweight="bold", pad=16
)
ax.legend(fontsize=13, framealpha=0.95, edgecolor="#e5e7eb", loc="upper right")
ax.grid(axis="y", alpha=0.20, lw=0.6)
ax.grid(axis="x", alpha=0)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT / "ticket_size_evolution.png", bbox_inches="tight")
plt.show()
print("Saved -> output/ticket_size_evolution.png")

## Chart 2 — Digital Payment Value Share (Stacked Area)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 11))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# Single line: UPI share % 
ax.plot(df["date"], df["upi_share_pct"], color="#1d4ed8", lw=3.0, zorder=4)
ax.fill_between(df["date"], df["upi_share_pct"], alpha=0.10, color="#3b82f6")

# Set limits early
y_top = max(df["upi_share_pct"].max() * 1.30, 15)
ax.set_xlim(df["date"].min(), df["date"].max())
ax.set_ylim(0, y_top)

# 10% reference line
ax.axhline(10, color="#dc2626", lw=1.6, ls="--", alpha=0.80, zorder=3)
ax.text(
    df["date"].min() + pd.DateOffset(months=1), 10.35,
    "10% milestone",
    fontsize=14, color="#dc2626", va="bottom", fontweight="bold"
)

# Crossover annotation
if crossover_10pct != "Not yet":
    co_date = pd.to_datetime(crossover_10pct + "-01")
    co_row  = df[df["date"] == co_date]
    if not co_row.empty:
        co_y = float(co_row["upi_share_pct"].iloc[0])
        ax.annotate(
            f"UPI crosses 10%\n({crossover_10pct})",
            xy=(co_date, co_y),
            xytext=(18, 36), textcoords="offset points",
            fontsize=13, color="#1d4ed8", ha="left",
            arrowprops=dict(arrowstyle="->", color="#1d4ed8", lw=1.4,
                            connectionstyle="arc3,rad=-0.2"),
            bbox=dict(boxstyle="round,pad=0.5", fc="#eff6ff",
                      ec="#3b82f6", alpha=0.96, lw=1.2)
        )

# Current value annotation
last_upi = df.iloc[-1]
ax.annotate(
    f"{last_upi['upi_share_pct']:.1f}% ({last_upi['month_year']})",
    xy=(last_upi["date"], last_upi["upi_share_pct"]),
    xytext=(-22, 28), textcoords="offset points",
    fontsize=14, color="#1d4ed8", fontweight="bold", ha="right",
    arrowprops=dict(arrowstyle="->", color="#1d4ed8", lw=1.4),
    bbox=dict(boxstyle="round,pad=0.5", fc="#eff6ff",
              ec="#3b82f6", alpha=0.96, lw=1.2)
)

# Axes
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax.tick_params(axis="x", rotation=30, labelsize=14)
ax.tick_params(axis="y", labelsize=14)
ax.set_ylabel("UPI Share of Digital Payment Value (%)", fontsize=16)
ax.set_title(
    "UPI's Share of India's Digital Payment Value —\n"
    "Crossing the 10% Threshold",
    fontsize=20, fontweight="bold", pad=16
)
ax.grid(axis="y", alpha=0.20, lw=0.6)
ax.grid(axis="x", alpha=0)
ax.spines[["top", "right"]].set_visible(False)

ax.text(0.985, 0.04,
        "By value — RTGS dominates; by volume UPI leads",
        transform=ax.transAxes, fontsize=12, ha="right", va="bottom",
        color="#374151", style="italic",
        bbox=dict(boxstyle="round,pad=0.3", fc="white",
                  ec="#e5e7eb", alpha=0.90, lw=0.7))

plt.tight_layout()
plt.savefig(OUTPUT / "digital_payment_share.png", bbox_inches="tight")
plt.show()
print("Saved -> output/digital_payment_share.png")

## Chart 3 — Value Race (Log Scale)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 11))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

ax.plot(df["date"], df["upi_value"],  color="#1d4ed8", lw=3.0, label="UPI")
ax.plot(df["date"], df["neft_value"], color="#16a34a", lw=3.0, label="NEFT")
ax.plot(df["date"], df["rtgs_value"], color="#64748b", lw=3.0, label="RTGS")

# Annotate UPI > NEFT crossover
if crossover_neft != "Not yet":
    co_date = pd.to_datetime(crossover_neft + "-01")
    co_row  = df[df["date"] == co_date]
    if not co_row.empty:
        co_y = float(co_row["upi_value"].iloc[0])
        ax.annotate(
            f"UPI crosses\nNEFT ({crossover_neft})",
            xy=(co_date, co_y),
            xytext=(34, 28), textcoords="offset points",
            fontsize=13, color="#1d4ed8",
            arrowprops=dict(arrowstyle="->", color="#1d4ed8", lw=1.4),
            bbox=dict(boxstyle="round,pad=0.45", fc="#eff6ff",
                      ec="#3b82f6", alpha=0.95, lw=1.1)
        )

# End-of-line labels with vertical separation to avoid overlap
label_items = [
    ("upi_value",  "#1d4ed8", "UPI"),
    ("neft_value", "#16a34a", "NEFT"),
    ("rtgs_value", "#64748b", "RTGS"),
]
last_r = df.iloc[-1]
label_vals = sorted(
    [(col, color, label, last_r[col]) for col, color, label in label_items],
    key=lambda x: x[3]
)
y_offsets = [-16, 0, 16]
for (col, color, label, val), yo in zip(label_vals, y_offsets):
    ax.annotate(label,
                xy=(last_r["date"], val),
                xytext=(6, yo), textcoords="offset points",
                fontsize=14, color=color, fontweight="bold", va="center")

ax.set_yscale("log")
ax.set_xlim(df["date"].min(), df["date"].max() + pd.DateOffset(months=3))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda y, _: f"₹{y/1e6:.1f}T" if y >= 1e6 else
                 f"₹{y/1e3:.0f}K cr" if y >= 1e3 else f"₹{y:.0f} cr"
))
ax.tick_params(axis="x", rotation=30, labelsize=14)
ax.tick_params(axis="y", labelsize=14)
ax.set_ylabel("Transaction Value (log scale)", fontsize=16)
ax.set_title("Digital Payment Value Race — UPI vs Legacy Rails",
             fontsize=20, fontweight="bold", pad=16)
ax.legend(fontsize=13, framealpha=0.95, edgecolor="#e5e7eb", loc="upper left")
ax.grid(axis="y", alpha=0.20, lw=0.6, which="both")
ax.grid(axis="x", alpha=0)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT / "value_race.png", bbox_inches="tight")
plt.show()
print("Saved -> output/value_race.png")

## Chart 4 — Network Effect

In [ ]:
# Use full upi dataframe to get all bank-count months
net = upi[["date", "month_year", "banks_live_on_upi", "volume_mn"]].dropna(
    subset=["banks_live_on_upi", "volume_mn"]
).sort_values("date").reset_index(drop=True)

# ── Find real inflection: first streak of 3+ months where MoM > 15% ──
net["vol_mom"] = net["volume_mn"].pct_change() * 100

streak_counts = []
count = 0
for v in (net["vol_mom"] > 15):
    count = count + 1 if v else 0
    streak_counts.append(count)
net["streak"] = streak_counts

first_3 = net[net["streak"] >= 3]
if not first_3.empty:
    # Walk back to the start of that streak
    inflection_idx = max(0, first_3.index[0] - 2)
    inflection_row = net.loc[inflection_idx]
else:
    # Fallback: month of highest single MoM jump
    inflection_row = net.loc[net["vol_mom"].idxmax()]

fig, ax1 = plt.subplots(figsize=(20, 12))
fig.patch.set_facecolor("white")
ax1.set_facecolor("white")

# Bars: UPI volume
bar_colors = ["#3b82f6" if v >= net["volume_mn"].median() else "#93c5fd"
              for v in net["volume_mn"]]
ax1.bar(net["date"], net["volume_mn"],
        color=bar_colors, alpha=0.70, width=20, label="UPI Volume (Mn)", zorder=2)
ax1.set_ylabel("Monthly Transaction Volume (Mn)", fontsize=16, color="#1d4ed8")
ax1.tick_params(axis="y", labelcolor="#1d4ed8", labelsize=14)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:,.0f}"))

# Set left axis limits so annotation can reference real y coordinates
ax1.set_xlim(net["date"].min() - pd.DateOffset(months=1),
             net["date"].max() + pd.DateOffset(months=1))
vol_top = net["volume_mn"].max() * 1.18
ax1.set_ylim(0, vol_top)

# Line: banks on UPI
ax2 = ax1.twinx()
ax2.plot(net["date"], net["banks_live_on_upi"],
         color="#dc2626", lw=2.8, label="Banks on UPI", zorder=3)
ax2.set_ylabel("Banks Live on UPI", fontsize=16, color="#dc2626")
ax2.tick_params(axis="y", labelcolor="#dc2626", labelsize=14)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{int(y)}"))
ax2.spines["top"].set_visible(False)

# Inflection annotation — above bars, clear of busy region
ax1.axvline(inflection_row["date"], color="#f59e0b",
            lw=2.0, ls="--", alpha=0.8, zorder=5)
ax1.annotate(
    f"Sustained growth phase begins\n"
    f"{inflection_row['month_year']}  (3+ months MoM > 15%)",
    xy=(inflection_row["date"], vol_top * 0.78),
    xytext=(20, 0), textcoords="offset points",
    fontsize=13, color="#92400e", ha="left", va="center",
    bbox=dict(boxstyle="round,pad=0.5", fc="#fef3c7",
              ec="#f59e0b", alpha=0.97, lw=1.2)
)

# Axes 
ax1.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax1.tick_params(axis="x", rotation=30, labelsize=14)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, fontsize=13, framealpha=0.95,
           edgecolor="#e5e7eb", loc="upper left")

ax1.set_title("The Network Effect — Banks on UPI vs Transaction Volume",
              fontsize=20, fontweight="bold", pad=16)
ax1.grid(axis="y", alpha=0.18, lw=0.6, zorder=0)
ax1.grid(axis="x", alpha=0)
ax1.spines["top"].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT / "network_effect.png", bbox_inches="tight")
plt.show()
print("Saved -> output/network_effect.png")
print(f"Inflection point identified: {inflection_row['month_year']}")

## Summary Statistics

In [ ]:
print("=" * 66)
print("  UPI DOMINANCE — KEY FINDINGS")
print("=" * 66)
print(f"\n  Date range      : {df['month_year'].min()} → {df['month_year'].max()}")
print(f"  Months analyzed : {len(df)}")
print()
print("TICKET SIZE")
print(f"  First month ({first['month_year']}) : ₹{ticket_first:,.0f}")
print(f"  Latest month ({last['month_year']}) : ₹{ticket_last:,.0f}")
print(f"  Change                 : {ticket_chg:+.0f}%")
print()
print("VALUE SHARE")
print(f"  UPI share first month  : {share_first:.1f}%")
print(f"  UPI share latest month : {share_last:.1f}%")
print(f"  NEFT share latest      : {last['neft_share_pct']:.1f}%")
print(f"  RTGS share latest      : {last['rtgs_share_pct']:.1f}%")
print()
print("MILESTONES")
print(f"  UPI first exceeded NEFT value  : {crossover_neft}")
print(f"  UPI first exceeded 10% share   : {crossover_10pct}")
print()
print("CAGR COMPARISON")
print(f"  UPI  value CAGR : {cagr_upi:+.1f}% / yr")
print(f"  NEFT value CAGR : {cagr_neft:+.1f}% / yr")
print(f"  RTGS value CAGR : {cagr_rtgs:+.1f}% / yr")
print()
print("CURRENT SPLIT (latest month) ")
print(f"  UPI  : {last['upi_share_pct']:.1f}%")
print(f"  NEFT : {last['neft_share_pct']:.1f}%")
print(f"  RTGS : {last['rtgs_share_pct']:.1f}%")
print("=" * 66)